# Notebook 01 — Explore the Baseline

This notebook builds the 3-feature LightGBM baseline entirely on synthetic data
so it runs in under three minutes.

Features (mirroring `src/utils/features.py`):
`TransactionAmt`, `time_since_creation`, `n_prior_transactions`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from phase01_lightgbm_baseline import LightGBMWrapper
from src.utils.features import BASELINE_FEATURES

rng = np.random.default_rng(42)
n = 4000
cards = rng.integers(0, 300, size=n)
fraud = (rng.random(n) < 0.035).astype(int)

raw = pd.DataFrame(
    {
        "TransactionID": np.arange(n),
        "card1": cards,
        "TransactionAmt": rng.lognormal(3.0, 1.0, size=n) * (1 + 2 * fraud),
        "TransactionDT": np.sort(rng.integers(0, 5_000_000, size=n)),
        "isFraud": fraud,
    }
)
print(raw.shape)
raw.head()

In [ ]:
def fit_baseline_feature_params(df, card_col="card1"):
    dt = df["TransactionDT"]
    return {"dt_min": float(dt.min()), "dt_max": float(dt.max())}


def prepare_baseline_features(df, params=None, card_col="card1"):
    # Fit on the training split only via params; never on train+test together.
    out = df.copy()
    if params is None:
        params = fit_baseline_feature_params(df, card_col=card_col)
    dt = out["TransactionDT"]
    dt_min, dt_max = params["dt_min"], params["dt_max"]
    if dt_max > dt_min:
        out["time_since_creation"] = ((dt - dt_min) / (dt_max - dt_min)).clip(0.0, 1.0)
    else:
        out["time_since_creation"] = 0.0
    out["n_prior_transactions"] = out.groupby(card_col).cumcount()
    return out


print(BASELINE_FEATURES)

## Train and evaluate

In [ ]:
raw_tr, raw_rest = train_test_split(raw, test_size=0.4, stratify=raw["isFraud"], random_state=42)
raw_val, raw_te = train_test_split(raw_rest, test_size=0.5, stratify=raw_rest["isFraud"], random_state=42)

# Fit the TransactionDT min/max on the training split only (leakage-free).
params = fit_baseline_feature_params(raw_tr)
train = prepare_baseline_features(raw_tr, params)
val = prepare_baseline_features(raw_val, params)
test = prepare_baseline_features(raw_te, params)

X_tr = train[BASELINE_FEATURES].to_numpy(dtype=float)
y_tr = train["isFraud"].to_numpy(dtype=int)
X_val = val[BASELINE_FEATURES].to_numpy(dtype=float)
y_val = val["isFraud"].to_numpy(dtype=int)
X_te = test[BASELINE_FEATURES].to_numpy(dtype=float)
y_te = test["isFraud"].to_numpy(dtype=int)

model = LightGBMWrapper()
k = max(1, int(np.ceil(0.01 * len(y_te))))
metrics = model.full_pipeline(X_tr, y_tr, X_val, y_val, X_te, y_te, k=k)

for name, value in metrics.items():
    print(f"{name:>18}: {value:.4f}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve

probs = model.calibrated_test_probs_
prec, rec, _ = precision_recall_curve(y_te, probs)
fpr, tpr, _ = roc_curve(y_te, probs)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(rec, prec, color="darkorange")
axes[0].set(xlabel="Recall", ylabel="Precision", title="PR curve (baseline)")
axes[1].plot(fpr, tpr, color="steelblue")
axes[1].plot([0, 1], [0, 1], ls="--", color="grey")
axes[1].set(xlabel="FPR", ylabel="TPR", title="ROC curve (baseline)")
fig.tight_layout()
fig.savefig("nb01_baseline_curves.png", dpi=120)
print("saved nb01_baseline_curves.png")

The baseline establishes the reference PR-AUC that any graph-aware model must beat.